# 🧭 Introduction to Supervised Machine Learning
### From finding structure to predicting answers

**Module 1 · Session 09 · lecture notebook**

Lecture notes for session 09, in notebook form. Run all cells (Runtime → Run all) and read top to bottom.
The charts are drawn from the nomad cities data used in sessions 06–08; the plotting code is folded away
and can be opened with *Show code*. The exercises are in part 1 and part 2.

<img src="https://raw.githubusercontent.com/aaubs/ds-master/main/media/M1_2026/sml09/nomad.jpg" width="520">

Contents
1. Where we are: from structure to prediction
2. The vocabulary: targets, losses, models
3. Honest evaluation: baselines, test sets, leakage
4. Classification and thresholds
5. Model families and interpretation
6. Practice: pipelines and tuning

In [ ]:
# @title Setup: data and chart style (run me first) { display-mode: "form" }
import warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore", message="Unknown solver options")   # harmless scipy noise from LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA, NMF
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, mean_absolute_error, precision_score,
                             r2_score, recall_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor, NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from matplotlib.colors import LinearSegmentedColormap

NAVY, LAV, GREY, ORANGE = "#211A52", "#C2C1CC", "#54616E", "#D2542A"   # AAU palette, same as the slides
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.edgecolor": GREY, "axes.labelcolor": GREY,
                     "xtick.color": GREY, "ytick.color": GREY, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#E6E6EC", "legend.frameon": False})

raw = pd.read_csv("https://sds-aau.github.io/SDS-master/M1/data/cities.csv")
cities = raw.copy()
cities.loc[cities["racism"] > 1, "racism"] = cities["racism"].median()   # Samara: 1.8e28 on a 0-1 scale
numeric = cities.select_dtypes("number")
OTHER_PRICES = ["cost_coworking", "cost_expat", "coffee_in_cafe", "cost_beer"]
FEATURES = [c for c in numeric.columns if c not in ["cost_nomad", *OTHER_PRICES]]
X, y = cities[FEATURES], cities["cost_nomad"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"{len(cities)} cities, {numeric.shape[1]} numeric columns. Ready.")

---
# 🗺️ 1 · Where we are

Sessions 06–08 worked with one table: 780 cities from Nomad List, with columns for cost, internet,
safety, freedom and nightlife.

| Session | Question asked of the table | Tool |
|---|---|---|
| 06 | Which few directions explain most of the variation? | PCA, UMAP, t-SNE |
| 07 | Which cities belong together? Which are the extremes? | k-means, archetypes |
| 08 | Which cities are most like this one? | similarity search |

All of these methods use only the columns, `X`. None of them is told what a right answer would be.

🖱️ The map below is interactive: hovering shows the city, and the chart can be zoomed.

In [ ]:
# @title The picture from sessions 06–07: PCA, coloured by k-means (interactive) { display-mode: "form" }
import altair as alt
Z = StandardScaler().fit_transform(numeric)
pcs = PCA(2).fit_transform(Z)
cluster = KMeans(4, n_init=10, random_state=42).fit(Z).labels_
view = cities[["place", "alpha-2", "region", "cost_nomad"]].assign(
    pc1=pcs[:, 0].round(2), pc2=pcs[:, 1].round(2), cluster=[f"cluster {k}" for k in cluster])
pick = alt.selection_point(fields=["cluster"], bind="legend")
dots = alt.Chart(view).mark_circle(size=45).encode(
    x=alt.X("pc1:Q", title="first component"), y=alt.Y("pc2:Q", title="second component"),
    color=alt.Color("cluster:N", scale=alt.Scale(range=[NAVY, LAV, GREY, ORANGE]), title="k-means"),
    opacity=alt.condition(pick, alt.value(0.85), alt.value(0.08)),
    tooltip=[alt.Tooltip("place:N", title="city"), alt.Tooltip("alpha-2:N", title="country"), "region:N",
             alt.Tooltip("cost_nomad:Q", title="USD / month", format=",.0f"), "cluster:N"],
).add_params(pick, alt.selection_interval(bind="scales"))
label = alt.Chart(view[view.place == "Aalborg"]).mark_text(dx=10, dy=-10, align="left", fontWeight="bold", color=NAVY
    ).encode(x="pc1:Q", y="pc2:Q", text="place:N")
(dots + label).properties(width=640, height=400, title="780 cities · hover for details · scroll to zoom · click the legend")

### 🧩 One more unsupervised idea: NMF

PCA gives directions that are positive and negative at the same time, which makes them hard to read.
**Non-negative matrix factorisation** only allows non-negative numbers, so each component becomes a *theme*
built from a handful of columns, and every city is a mix of those themes, like ingredients in a recipe.

In [ ]:
# @title Four latent themes in the nomad data { display-mode: "form" }
Xn = numeric.drop(columns=["cost_beer"])                        # identical to coffee_in_cafe in every row
scaled = MinMaxScaler().fit_transform(Xn)
nmf = NMF(4, init="nndsvda", random_state=0, max_iter=2000).fit(scaled)
W = nmf.transform(scaled)
H = pd.DataFrame(nmf.components_, columns=Xn.columns)
H = H.div(H.max(axis=1), axis=0)
label = {"nightlife": "Buzz & lifestyle", "fragile_states_index": "Fragile & restricted",
         "weed": "Cannabis (one column)", "peace_score": "Safe & open"}
names = [label.get(H.iloc[i].idxmax(), H.iloc[i].idxmax()) for i in range(4)]
feats = list(dict.fromkeys(f for i in range(4) for f in H.iloc[i].sort_values(ascending=False).index[:4]))
fig, ax = plt.subplots(figsize=(8.5, 3.4))
ax.imshow(H[feats].values, cmap=LinearSegmentedColormap.from_list("aau", ["#FFFFFF", LAV, NAVY]), aspect="auto")
ax.set_yticks(range(4), names); ax.set_xticks(range(len(feats)), [f.replace("_", " ") for f in feats], rotation=40, ha="right")
ax.grid(False); ax.set_title("what each theme is made of", color=NAVY)
plt.show()
for i, n in enumerate(names):
    print(f"{n:24s}", ", ".join(cities.loc[np.argsort(-W[:, i])[:3], "place"]))

The themes almost name themselves. One of them is simply *cannabis*: a single column strong enough to get a
theme of its own. The method has no notion of what is interesting.

🤔 *Question: which of these themes would predict cost best?*

None of the themes is about cost. NMF finds structure, but it is never asked about cost.

### How were those results judged?

By interpretability: whether the first component can be read, whether the city types make sense, whether
the recommendations are plausible. That is judgement, not a check against an answer key. Supervised
learning adds the answer key.

### 🙈 Hide one column

A new city appears on Nomad List. We know its internet speed, safety, freedom score, nightlife…
but nobody has reported what it **costs** to live there.

<img src="https://raw.githubusercontent.com/aaubs/ds-master/main/media/M1_2026/sml09/hide_column.jpg" width="480">

`cost_nomad` (monthly cost for a remote worker, in USD) becomes the thing to predict: **`y`**.
Everything else stays the features: **`X`**. Same table, one column promoted to target.

| | Unsupervised | Supervised |
|---|---|---|
| Input | `X` only | `X` **and** `y` |
| Question | What structure is in here? | Given `X`, what is `y`? |
| Output | components, clusters, neighbours | a prediction for a new row |
| Judged by | interpretability, usefulness | error on rows the model has **not seen** |

A right answer makes error measurable, and measurable error is what allows models to be compared, tuned
and checked for cheating.

### 🏘️ The similarity search is already a supervised model

Session 08: find the five cities most like Aalborg, with the cost columns hidden.

🤔 *Question: what would these neighbours suggest Aalborg costs?*

In [ ]:
Zx = StandardScaler().fit_transform(X)
aalborg = cities.index[cities["place"] == "Aalborg"][0]
_, idx = NearestNeighbors(n_neighbors=6).fit(Zx).kneighbors(Zx[[aalborg]])
neighbours = [i for i in idx[0] if i != aalborg][:5]
display(cities.loc[neighbours, ["place", "alpha-2", "cost_nomad"]])
print(f"average of the neighbours: {cities.loc[neighbours, 'cost_nomad'].mean():,.0f}")
print(f"Aalborg, actual:           {cities.loc[aalborg, 'cost_nomad']:,.0f}")

That average **is** a prediction. Averaging the neighbours' `y` is called **k-nearest-neighbours regression**,
and scikit-learn has it as one object:

In [ ]:
others = cities.index != aalborg
knn = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5)).fit(X[others], y[others])
knn.predict(X.loc[[aalborg]])

Off by about 840 USD. Is that good or bad? We can't say yet: we need something to compare with.

### 🏷️ Clusters are not labels

The k-means clusters were built on all the columns, cost included. Do they answer *is this city expensive?*

In [ ]:
# @title Share of cities above 3,000 USD per cluster { display-mode: "form" }
pd.crosstab(cluster, cities["cost_nomad"] > 3000, normalize="index")[True].rename("share expensive").round(2).to_frame().T

The clusters are related to cost, but one of them is close to a coin flip. A question about cost needs a model
that is given cost, which is what supervision means.

What carries over from unsupervised learning: scaling (`StandardScaler`), distance and similarity (kNN is
the session 08 search), components as features, and the `fit` / `transform` pattern, now with `predict`.

What is new: a target, a loss function, and a test set.

---
# 📖 2 · The vocabulary

1. **Unsupervised**: unlabelled data, find structure *(sessions 06–08)*
2. **Supervised**: labelled data, predict outcomes *(today)*
3. **Self-supervised**: make labels from the data itself: hide the next word, predict it *(how LLMs are pre-trained)*
4. **Reinforcement**: learn from rewards while interacting with an environment

The type of `y` decides the task: a number → **regression** (monthly cost); a category → **classification**
(expensive above 3,000 USD, or not). Same features, same workflow; different loss and metrics.

Key terms: training data, features and labels, model, loss, evaluation, and generalisation (doing well on
the next city, not the last one).

We look for parameters $\theta$ such that $y \approx f(X;\theta)$ **for cities the model has not seen**.
"$\approx$" needs a definition: the loss.

| Mean squared error (regression) | Cross-entropy (classification) |
|---|---|
| $\text{MSE} = \frac{1}{n}\sum_i (y_i - \hat y_i)^2$ | $H(p,q) = -\sum_k p_k \log q_k$ |
| Squaring punishes big misses: 2,000 off counts 100× more than 200 off | Only the probability of the right class counts; confident mistakes cost most: $-\log(0.01) \approx 4.6$ |

In practice RMSE or MAE is reported, because both are in the units of `y` (USD, not USD²).

In [ ]:
y_true = np.array([3.5, 2.1, 4.0, 5.5, 6.1]); y_hat = np.array([3.8, 2.0, 4.2, 5.0, 6.0])
print("MSE:", np.mean((y_true - y_hat) ** 2))
p = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]]); q = np.array([[.8, .1, .1], [.2, .7, .1], [.1, .2, .7]])
print("cross-entropy:", round(-np.mean(np.sum(p * np.log(np.clip(q, 1e-12, 1)), axis=1)), 3))

---
# 📏 3 · Honest evaluation

### First, a baseline
Before any model: predict the **average cost** for every city. Every model has to beat that.

In [ ]:
baseline = mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))
print(f"baseline MAE on held-out cities: {baseline:,.0f} USD")

For Aalborg, the average (about 2,330) is off by almost 1,900; the neighbours were off by about 840.
A model's error only means something next to a baseline.

### Train-test split
The model trains on 80 % of the rows; the other 20 % are held back. The test score estimates performance
on the next city. Five models, each scored on its training rows and on the held-back rows:

In [ ]:
# @title Train vs test error, five models { display-mode: "form" }
models = {"1-nearest neighbour": make_pipeline(StandardScaler(), KNeighborsRegressor(1)),
          "tree, no depth limit": DecisionTreeRegressor(random_state=0),
          "linear regression": make_pipeline(StandardScaler(), LinearRegression()),
          "tree, depth 3": DecisionTreeRegressor(max_depth=3, random_state=0),
          "random forest": RandomForestRegressor(300, random_state=0)}
rows = []
for n, m in models.items():
    m.fit(X_train, y_train)
    rows.append((n, mean_absolute_error(y_train, m.predict(X_train)), mean_absolute_error(y_test, m.predict(X_test))))
fig, ax = plt.subplots(figsize=(8, 3.8))
for i, (n, tr, te) in enumerate(rows[::-1]):
    ax.plot([tr, te], [i, i], color=LAV, lw=3, zorder=1)
    ax.scatter(tr, i, color=NAVY, s=60, zorder=2, label="train" if i == 0 else None)
    ax.scatter(te, i, color=ORANGE, s=60, zorder=2, label="test" if i == 0 else None)
    ax.text(te + 18, i, f"{te:.0f}", va="center", color=ORANGE); ax.text(tr - 18, i, f"{tr:.0f}", va="center", ha="right", color=NAVY)
ax.axvline(baseline, color=GREY, ls="--", lw=1); ax.text(baseline + 12, 2, f"baseline\n{baseline:.0f}", color=GREY, va="center")
ax.set_yticks(range(len(rows)), [r[0] for r in rows[::-1]]); ax.set(xlabel="mean absolute error (USD per month)", xlim=(-90, 1000))
ax.grid(axis="y", visible=False); ax.legend(loc="lower center", bbox_to_anchor=(0.45, 1.0), ncol=2)
plt.show()

The unlimited tree and 1-NN are near-perfect on training and among the worst on test: they memorised the
training cities. The ranking by training error is almost the reverse of the ranking by test error.

### Cross-validation
One split is one roll of the dice. 5-fold CV trains on four folds, scores the fifth, and rotates: five scores,
an average and a spread. It gives a more stable estimate; it does not reduce overfitting by itself.

In [ ]:
for n in ["linear regression", "random forest"]:
    s = -cross_val_score(models[n], X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
    print(f"{n:18s} mean {s.mean():,.0f}   spread {s.std():,.0f}")

### ⚖️ Bias and variance

<img src="https://raw.githubusercontent.com/aaubs/ds-master/main/media/M1_2026/sml09/memorising.jpg" width="460">

- **Bias**: too simple to capture the pattern (underfitting). *Depth-3 tree.*
- **Variance**: so flexible it fits the noise (overfitting). *Unlimited tree, 1-NN.*
- **Irreducible error**: noise no model can explain.

The aim is the complexity where test error is lowest. The trade-off measured for kNN, with $k$ as the knob:

In [ ]:
# @title k-nearest neighbours: error against k { display-mode: "form" }
ks = [1, 2, 3, 5, 7, 10, 15, 20, 30, 50, 75, 100, 200, 400]
tr, te = [], []
for k in ks:
    m = make_pipeline(StandardScaler(), KNeighborsRegressor(k)).fit(X_train, y_train)
    tr.append(mean_absolute_error(y_train, m.predict(X_train))); te.append(mean_absolute_error(y_test, m.predict(X_test)))
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(ks, tr, marker="o", ms=4, color=NAVY, label="train"); ax.plot(ks, te, marker="o", ms=4, color=ORANGE, label="test")
ax.axhline(baseline, color=GREY, ls="--", lw=1)
ax.text(1.25, 130, "variance:\nmemorises", color=NAVY, fontsize=9); ax.text(90, 520, "bias:\ntoo smooth", color=NAVY, fontsize=9)
ax.set(xscale="log", xlabel="k (log scale)", ylabel="MAE (USD)"); ax.legend(loc="center right")
plt.show()

$$E\big[(y - \hat f(x))^2\big] = \text{Var}\big(\hat f(x)\big) + \text{Bias}\big(\hat f(x)\big)^2 + \text{Var}(\epsilon)$$

Draw many training sets, fit the model on each, predict Aalborg: **variance** is how much those predictions
jump around, **bias** is how far their average sits from the truth.

Regularisation trades a little bias for a lot less variance: Lasso (L1) pushes coefficients to zero,
Ridge (L2) shrinks them all, Elastic net mixes both. Note that scikit-learn calls the strength $\lambda$ `alpha`,
and the mix `l1_ratio`.

### 🚰 Leakage: is the feature available at prediction time?

<img src="https://raw.githubusercontent.com/aaubs/ds-master/main/media/M1_2026/sml09/leakage.jpg" width="440">

🤔 *Question: for a city without a cost report, would the price of a coworking desk be known?*

In [ ]:
def linear_r2(cols):
    a, b, c, d = train_test_split(cities[cols], y, test_size=0.2, random_state=42)
    return r2_score(d, make_pipeline(StandardScaler(), LinearRegression()).fit(a, c).predict(b))
print(f"with the other price columns:    R² = {linear_r2(FEATURES + OTHER_PRICES):.2f}")
print(f"without the other price columns: R² = {linear_r2(FEATURES):.2f}")
print("coffee equals beer in every row:", (cities.coffee_in_cafe == cities.cost_beer).all())

A feature is only legitimate if it exists when the prediction is made. Leakage makes a model look excellent
in the notebook and fail the day it is used. (Also: Samara's `racism` score was 1.8 × 10²⁸ on a 0–1 scale. It
was found because one cross-validation fold blew up. A wildly bad fold is often a data problem.)

---
# 🎯 4 · Classification: is this city expensive?

`y = 1` if `cost_nomad` is above 3,000 USD. 28 % of cities are expensive.

The comparison that matters is a model that always answers "not expensive".

In [ ]:
yc = (cities["cost_nomad"] > 3000).astype(int)
Xa, Xb, ya, yb = train_test_split(X, yc, test_size=0.2, random_state=42, stratify=yc)
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xa, ya)
proba = logit.predict_proba(Xb)[:, 1]
print(f"logistic regression accuracy: {accuracy_score(yb, proba >= 0.5):.2f}")
print(f"'never expensive' accuracy:   {1 - yb.mean():.2f}")
print("confusion matrix (rows = actual cheap/expensive):\n", confusion_matrix(yb, proba >= 0.5))
print(f"precision {precision_score(yb, proba >= 0.5):.2f} · recall {recall_score(yb, proba >= 0.5):.2f}")

With unbalanced classes, accuracy hides most of what matters. Precision: of the cities flagged as expensive,
how many are? Recall: of the expensive cities, how many are flagged? The model outputs a probability, and
the 0.5 cut-off is a choice:

In [ ]:
# @title Same model, different cut-offs { display-mode: "form" }
ts = np.arange(0.1, 0.81, 0.05)
curves = {"precision": [precision_score(yb, proba >= t, zero_division=0) for t in ts],
          "recall": [recall_score(yb, proba >= t) for t in ts],
          "accuracy": [accuracy_score(yb, proba >= t) for t in ts]}
fig, ax = plt.subplots(figsize=(7, 3.8))
for k, c in zip(curves, [NAVY, ORANGE, GREY]):
    ax.plot(ts, curves[k], marker="o", ms=4, color=c, label=k)
for t in (0.3, 0.5):
    ax.axvline(t, color=LAV, lw=6, alpha=0.5, zorder=0)
ax.set(xlabel="threshold", ylabel="score", ylim=(0, 1.05)); ax.legend(loc="lower left")
plt.show()
print(f"ROC-AUC {roc_auc_score(yb, proba):.2f}: how well the probabilities rank expensive above cheap")

🤔 *Question: which threshold is right?*

<img src="https://raw.githubusercontent.com/aaubs/ds-master/main/media/M1_2026/sml09/threshold.jpg" width="440">

It depends on what each mistake costs. A **miss**: a client is promised a budget that won't hold. A **false
alarm**: an analyst checks a city that was fine. Different costs, different cut-off. *(Session 10.)*

---
# 🌳 5 · Model families and interpretation

- **Linear models**: linear and logistic regression, plus Ridge, Lasso, Elastic net. Fast, stable, readable;
  they miss curves and interactions unless these are built in as features.
- **Neighbour models**: kNN. Small `k` = high variance, large `k` = high bias; needs scaling.
- **Trees**: recursive splits. **Random forests** average many deep trees; **gradient boosting** (XGBoost) adds
  shallow trees that fix each other's errors *(session 10)*.
- **Neural networks**: MLPs, RNNs, transformers, GNNs *(module 4)*. On tabular business data, a tuned tree
  ensemble is usually the stronger baseline.

A real tree, depth 2, predicting "expensive" for all 780 cities:

In [ ]:
# @title Depth-2 decision tree { display-mode: "form" }
tree = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X, yc)
fig, ax = plt.subplots(figsize=(10, 4.2))
plot_tree(tree, feature_names=FEATURES, class_names=["cheap", "expensive"], filled=True, rounded=True,
          impurity=False, proportion=True, ax=ax)
plt.show()

A lower press freedom index means a freer press.

### What drives the prediction?
Coefficients (effect of a feature, **others held fixed**), feature importance (trees), and SHAP (one
prediction at a time).

🤔 *Question: `life_score` correlates positively with cost. What sign should its coefficient have?*

In [ ]:
# @title Correlation vs coefficient { display-mode: "form" }
lin = models["linear regression"]
coef = pd.Series(lin[-1].coef_, index=FEATURES); corr = X_train.corrwith(y_train)
show = ["fragile_states_index", "life_score", "freedom_score", "press_freedom_index", "peace_score", "internet_speed"]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
for ax, s, title in [(axes[0], corr[show], "correlation with cost"), (axes[1], coef[show], "coefficient (USD per 1 SD)")]:
    ax.barh(show[::-1], s[show[::-1]], color=[ORANGE if v < 0 else NAVY for v in s[show[::-1]]])
    ax.axvline(0, color=GREY, lw=0.8); ax.set_title(title, color=NAVY); ax.grid(axis="y", visible=False)
plt.show()

Holding the others fixed, what's left of `life_score` points the other way: it overlaps with freedom, peace and
stability. All of these tools describe **the model, not the world**. The fragile states index predicts cost; it
doesn't cause it. Prediction versus causation is one of the module's learning objectives.

---
# 🧪 6 · Practice: pipelines and tuning

Preprocessing belongs inside a pipeline, so that the scaler learns from the training rows only. Hyperparameters
are chosen with cross-validation on the training set, and the test set is used once, at the end.

In [ ]:
search = GridSearchCV(make_pipeline(StandardScaler(), KNeighborsRegressor()),
                      {"kneighborsregressor__n_neighbors": [1, 3, 5, 10, 20]},
                      cv=5, scoring="neg_mean_absolute_error").fit(X_train, y_train)
print("best k:", search.best_params_["kneighborsregressor__n_neighbors"],
      f"| CV MAE {-search.best_score_:,.0f} | test MAE {mean_absolute_error(y_test, search.predict(X_test)):,.0f}")

---
# 📋 Wrap-up

- Unsupervised learning asks what structure is in `X`. Supervised learning asks what `y` is, given `X`.
- The difference is the answer key, which makes error measurable.
- Beat a baseline, measure on unseen data, check for leakage.
- Balance bias and variance; regularisation and ensembles help.
- Pick metrics for the decision: accuracy is rarely enough.
- Interpretation describes the model, not causation.

Exercises (save a copy in Drive before starting):
- [Part 1 · From finding structure to predicting answers](https://colab.research.google.com/github/aaubs/ds-master/blob/main/notebooks/M1_09_1_from_uml_to_sml.ipynb): every number above, with exercises and solutions
- [Part 2 · Predicting Airbnb prices in Copenhagen](https://colab.research.google.com/github/aaubs/ds-master/blob/main/notebooks/M1_09_2_airbnb_price_prediction.ipynb): the same workflow on messy real data

*Session 10:* gradient boosting, temporal splits, calibration, thresholds from business costs.